Notebook to plot how the computed albedo in each model timestep of the TOP experiments (parallel algorithm output) correlates with other ice output variables, for each ice category. This shows that jumps in albedo correlate with most clearly with the sign of ice surface temperature. Produces `top_albedo_dependence.png`. 

In [ ]:
from AOSCMcoupling import NEMOPreprocessor
import xarray as xr
import proplot as pplt
import pandas as pd
from pathlib import Path
import numpy as np
import warnings

In [ ]:
dates = pd.date_range("2020-04-12 00:00", "2020-04-18 22:00", freq="2h")
max_iters = 30
tldir = Path("/home/valentina/dev/aoscm/experiments/output/top_ensemble_full")
assert tldir.is_dir()

In [ ]:
nemo_preproc = [NEMOPreprocessor(date).preprocess for date in dates]

In [ ]:
def get_date_dirs(dates: pd.DatetimeIndex) -> list[Path]:
    date_dirs = [tldir / f"{date.date()}_{date.hour:02}" for date in dates]
    for date_dir in date_dirs:
        assert date_dir.is_dir()
    return date_dirs

In [ ]:
def load_all_iterates(
    dates: pd.DatetimeIndex,
    file_name: str,
    preprocess: list[callable],
    iter_range: range,
) -> xr.Dataset:
    date_dirs = get_date_dirs(dates)
    swr_dim = xr.DataArray(np.array(iter_range), dims="swr_iterate")
    start_date_dim = xr.DataArray(dates, dims="start_date")
    swrs = []
    for date_dir, preproc in zip(date_dirs, preprocess):
        files = [date_dir / f"iter_{iter + 1}/{file_name}" for iter in iter_range]
        iterates = [xr.open_mfdataset(str(file), preprocess=preproc) for file in files]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            swr = xr.concat(iterates, swr_dim)
        swrs.append(swr)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        concatenated_ds = xr.concat(swrs, start_date_dim)
    return concatenated_ds

In [ ]:
icemod = load_all_iterates(dates, "*icemod*.nc", nemo_preproc, range(1))

In [ ]:
ttop = icemod.icettop_cat.stack(z=("start_date", "swr_iterate", "time"))
ithic = icemod.icethic_cat.stack(z=("start_date", "swr_iterate", "time"))
sthic = icemod.snwthic_cat.stack(z=("start_date", "swr_iterate", "time"))
alb = icemod.icealb_cat.stack(z=("start_date", "swr_iterate", "time"))
ttop_sign = xr.where(ttop < 0, -1, 0)

In [ ]:
fig, axs = pplt.subplots(nrows=4, ncols=5, sharey=2, sharex=0, spanx=True, spany=True)
for cat in range(1, 6):
    ax = axs[0, cat - 1]
    ax.scatter(ttop.sel(ncatice=cat), alb.sel(ncatice=cat), color="k")
    ax.format(xlabel="Ice Surface Temperature [°C]", ylabel="Albedo [0-1]")

    ax = axs[1, cat - 1]
    ax.scatter(ttop_sign.sel(ncatice=cat), alb.sel(ncatice=cat), color="k")
    ax.format(xlabel="Sign of Ice Surface Temperature", ylabel="Albedo [0-1]", xticks=[-1, 0])

    ax = axs[2, cat - 1]
    ax.scatter(ithic.sel(ncatice=cat), alb.sel(ncatice=cat), color="k")
    ax.format(xlabel="Ice Thickness [m]", ylabel="Albedo [0-1]")

    ax = axs[3, cat - 1]
    ax.scatter(sthic.sel(ncatice=cat), alb.sel(ncatice=cat), color="k")
    ax.format(xlabel="Snow Thickness [m]", ylabel="Albedo [0-1]")

axs.format(abc="a)", toplabels=[f"Category {cat}" for cat in range(1, 6)], abcloc="ll")
fig.savefig("top_albedo_dependence.png", dpi=300)